# 15 - Gold 4: mart_network_daily (the KPI mart)
**Grain:** one row per (`event_date`, `region_id`). Business key is unique at that grain.

| KPI | Business definition |
|---|---|
| `availability_pct` | share of readings that are NOT in an outage, `100 * (readings - outage_readings) / readings`. An outage reading is one whose current status (after CDC) is FAILED |
| `avg_packet_loss_pct` | average `packet_loss_pct` over all readings. Use `packet_loss_sum / readings` when rolling up |
| `outage_minutes` | sum of per-reading outage minutes (time to the tower's next reading while FAILED). Open outages add 0 and are counted in `open_outage_readings`. Minutes belong to the date of the reading that starts them |
| `avg_latency_ms_excl_outliers` | average latency ignoring the 35 readings flagged as extreme (kept in Silver, excluded here) |
| `affected_customers` | distinct `operator_id` on outage readings (assumption: the closest customer key in the data) |

**Idempotent:** the mart is recomputed from the fact and merged by key, so running the same input again cannot double count.

In [0]:
BRONZE = "capstone6_telecom.bronze"
SILVER = "capstone6_telecom.silver"
GOLD   = "capstone6_telecom.gold"
cat    = GOLD   # this notebook writes to the gold schema

## 1. Aggregate from the fact

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW _mart_agg AS
SELECT f.event_date, f.region_id, f.region_key,
       COUNT(*) AS readings,
       SUM(CASE WHEN f.is_outage_reading THEN 1 ELSE 0 END) AS outage_readings,
       ROUND(100.0 * (COUNT(*) - SUM(CASE WHEN f.is_outage_reading THEN 1 ELSE 0 END)) / COUNT(*), 2) AS availability_pct,
       SUM(f.packet_loss_pct) AS packet_loss_sum,
       ROUND(AVG(f.packet_loss_pct), 3) AS avg_packet_loss_pct,
       MAX(f.packet_loss_pct) AS max_packet_loss_pct,
       ROUND(AVG(CASE WHEN NOT f.latency_outlier_flag THEN f.latency_ms END), 2) AS avg_latency_ms_excl_outliers,
       SUM(CASE WHEN f.latency_outlier_flag THEN 1 ELSE 0 END) AS latency_outlier_readings,
       SUM(f.outage_minutes) AS outage_minutes,
       SUM(CASE WHEN f.is_open_outage THEN 1 ELSE 0 END) AS open_outage_readings,
       COUNT(DISTINCT CASE WHEN f.is_outage_reading THEN f.tower_id END) AS impacted_towers,
       COUNT(DISTINCT CASE WHEN f.is_outage_reading THEN f.operator_id END) AS affected_customers,
       SUM(CASE WHEN f.has_alert THEN 1 ELSE 0 END) AS alert_readings
FROM {cat}.fact_network_activity f
GROUP BY f.event_date, f.region_id, f.region_key
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW _mart_src AS
SELECT *, xxhash64(readings, outage_readings, packet_loss_sum, outage_minutes, impacted_towers,
                   affected_customers, alert_readings, latency_outlier_readings, open_outage_readings) AS metrics_hash,
       current_timestamp() AS updated_at
FROM _mart_agg
""")

## 2. Create once, then MERGE by (event_date, region_id)

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {cat}.mart_network_daily AS SELECT * FROM _mart_src WHERE 1 = 0")

spark.sql(f"""
MERGE INTO {cat}.mart_network_daily t
USING _mart_src s
ON t.event_date = s.event_date AND t.region_id = s.region_id
WHEN MATCHED AND t.metrics_hash <> s.metrics_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

## 3. Overall daily view (safe roll-up: sums counts first, then divides)

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {cat}.mart_network_daily_overall AS
SELECT event_date, SUM(readings) AS readings, SUM(outage_readings) AS outage_readings,
       ROUND(100.0 * (SUM(readings) - SUM(outage_readings)) / SUM(readings), 2) AS availability_pct,
       ROUND(SUM(packet_loss_sum) / SUM(readings), 3) AS avg_packet_loss_pct,
       SUM(outage_minutes) AS outage_minutes
FROM {cat}.mart_network_daily
GROUP BY event_date""")

display(spark.sql(f"SELECT * FROM {cat}.mart_network_daily_overall ORDER BY event_date"))

## 4. Checks

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS region_days,
       COUNT(*) - COUNT(DISTINCT event_date, region_id) AS duplicate_keys,   -- expect 0
       SUM(readings) AS total_readings,                                       -- expect 11,500
       SUM(outage_readings) AS total_outage_readings                          -- expect 2,755
FROM {cat}.mart_network_daily"""))
display(spark.sql(f"SELECT * FROM {cat}.mart_network_daily ORDER BY outage_minutes DESC LIMIT 10"))